# Etapa 1 — EDA de SpaceLAB

Análisis reproducible de imágenes satelitales **ópticas RGB** y anotaciones YOLO.
No requiere GPU. No entrena modelos, crea particiones, corrige anotaciones ni modifica imágenes.
Incluye inventario y calidad, distribución de clases y dimensiones, además de
evidencia descriptiva sobre escala de objetos, densidad y riesgo de fuga.
No se redimensionan los datos; Matplotlib solamente adapta la representación en pantalla.

**Uso en Colab:** abrir este notebook desde GitHub, seleccionar CPU, disponer del dataset
descomprimido en `/content` o Drive y ejecutar de arriba abajo. Las imágenes no están en GitHub.
Si se usa Drive, montarlo con la celda opcional siguiente y configurar `DATASET_ROOT`.
La ruta puede señalar la carpeta externa o la anidada. No se descarga ni descomprime nada automáticamente.

**Protocolo:** auditoría estructural de `train`, `test` y `oculto`; gráficos y decisiones
descriptivas basados por defecto en `train`. Se leen las etiquetas de `test` únicamente para
integridad, no para ajustar modelos, umbrales ni hiperparámetros. `oculto` no tiene verdad terreno:
no se le asignan ceros ni se buscan o infieren etiquetas. Una auditoría de integridad no convierte
un test ya utilizado en un nuevo test independiente.

La métrica del reto es `0.7 × mAP@0.5:0.95 + 0.3 × F1macro@0.5`.
No puede calcularse rendimiento de detección sin predicciones. Los baselines del kit validan y
ajustan el umbral en `test`; si el equipo quiere una comprobación final independiente deberá
definir más adelante validación dentro de `train`, sin ejecutar aún esa decisión.

**Fundamento:** [E], cap. 4, métricas, baseline, preparación y evaluación;
[L], caps. 2, 5 y 6, lectura, visualización, calidad, sesgos, partición y resolución.
Las comprobaciones de cajas y hashes son adaptaciones de este proyecto, no técnicas atribuidas a los libros.
Las fuentes bibliográficas públicas permiten verificar capítulos/temas; no se afirma haber consultado
íntegramente las ediciones de pago. Bibliografía al final.


In [ ]:
# Opcional en Colab: descomentar si los datos están en Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')


## 1. Entorno, rutas y trazabilidad

**Pregunta:** ¿qué datos y versiones analiza esta ejecución?
**Método:** semilla fija para muestras visuales, rutas ordenadas, búsqueda acotada de carpetas
con `data.yaml` e `images/train`. Una ruta explícita tiene prioridad y debe ser válida;
si hay varias raíces automáticas se solicita elegir una para evitar mezclar datasets.
Se registran versiones y huellas SHA-256. PyYAML se usa para validar el contrato del dataset.
Se instalan únicamente dependencias ausentes; para fijar el entorno local se suministra
`requirements-eda.txt`, y el manifiesto conserva las versiones exactas utilizadas.


In [1]:
import importlib.util
import subprocess
import sys

AUTO_INSTALL = True  # False en un entorno offline ya preparado.
DEPENDENCIES = {
    'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4',
    'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<0.14',
    'PIL': 'Pillow>=10,<14', 'cv2': 'opencv-python-headless>=4.8,<6',
    'yaml': 'PyYAML>=6,<7',
}
missing = [package for module, package in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not AUTO_INSTALL:
        raise RuntimeError(f'Dependencias ausentes: {missing}. Instale requirements-eda.txt.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import hashlib
import json
import os
import platform
import re
import time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from PIL import Image

SEED = 42
TOL = 1e-6  # Tolerancia absoluta en coordenadas normalizadas; no se recortan cajas.
DATASET_ROOT = os.environ.get('SPACELAB_DATASET') or None
# Ejemplos: '/content/drive/MyDrive/spacelab_dataset' o '../spacelab_dataset'
EDA_SPLITS = ('train',)
HASH_IMAGES = True  # SHA de archivo y píxeles nativos; requiere leer todo el dataset.
EXPORT = True
OUTPUT_DIR = None  # Por defecto <repo>/reports/eda o <cwd>/reports/eda.
SPLITS = ('train', 'test', 'oculto')
CLASS_NAMES = ['small_boat', 'cargo_ship', 'fishing_vessel', 'warship', 'work_vessel']
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
EXPECTED = {'train': (5926, 22155), 'test': (801, 3964), 'oculto': (1743, None)}
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12})
pd.set_option('display.max_columns', 30)

def locate_dataset(explicit=None, bases=None):
    examined = []
    bases = list(bases) if bases is not None else [
        Path.cwd(), *list(Path.cwd().parents)[:2], Path('/content'),
        Path('/content/SpaceLAB'), Path('/content/drive/MyDrive'),
    ]
    if explicit:
        bases = [Path(explicit).expanduser()]
    candidates = []
    for base in bases:
        for suffix in ('', 'spacelab_dataset', 'spacelab_dataset/spacelab_dataset'):
            candidate = (base / suffix).resolve()
            if candidate in examined:
                continue
            examined.append(candidate)
            if (candidate / 'data.yaml').is_file() and (candidate / 'images/train').is_dir():
                candidates.append(candidate)
    if len(candidates) != 1:
        raise RuntimeError(
            f'Se encontraron {len(candidates)} datasets. Configure DATASET_ROOT explícitamente. '
            'Debe existir data.yaml e images/train. Rutas examinadas:\n'
            + '\n'.join(str(p) for p in examined)
        )
    return candidates[0], examined

DATA_ROOT, EXAMINED = locate_dataset(DATASET_ROOT)
repo_candidates = [Path.cwd(), *list(Path.cwd().parents)[:3]]
REPO_ROOT = next((p for p in repo_candidates if (p / 'kit_participantes').is_dir()), Path.cwd())
OUT = Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR else REPO_ROOT / 'reports/eda'
if OUT == DATA_ROOT or DATA_ROOT in OUT.parents:
    raise ValueError('OUTPUT_DIR debe estar fuera del dataset para conservar los originales.')
if EXPORT:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / 'figures').mkdir(exist_ok=True)

def save_figure(name):
    plt.tight_layout()
    if EXPORT:
        plt.savefig(OUT / 'figures' / f'{name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

config = yaml.safe_load((DATA_ROOT / 'data.yaml').read_text(encoding='utf-8-sig'))
names = config.get('names', {})
if isinstance(names, dict):
    names = [names.get(i, names.get(str(i))) for i in range(5)]
if names != CLASS_NAMES or config.get('nc') != 5:
    raise ValueError(f'Clases diferentes de las oficiales: {config}')
if config.get('train') != 'images/train' or config.get('val') != 'images/test':
    raise ValueError(f'Las rutas del YAML difieren del kit oficial; revise: {config}')
VERSIONS = {p: version(p) for p in ['numpy', 'pandas', 'matplotlib', 'seaborn', 'Pillow', 'PyYAML']}
VERSIONS['opencv'] = cv2.__version__
print('Dataset:', DATA_ROOT, '\nSalida:', OUT, '\nPython:', platform.python_version())
display(pd.Series(VERSIONS, name='versión').to_frame())
display(pd.DataFrame({'class_id': range(5), 'clase': CLASS_NAMES}))
print('data.yaml:', config)


Dataset: C:\Users\jlaur\SpaceLAB\spacelab_dataset\spacelab_dataset 
Salida: c:\Users\jlaur\SpaceLAB\reports\eda 
Python: 3.13.14


,versión
numpy,2.5.2
pandas,3.0.5
matplotlib,3.11.1
seaborn,0.13.2
Pillow,12.3.0
PyYAML,6.0.3
opencv,5.0.0


,class_id,clase
0,0,small_boat
1,1,cargo_ship
2,2,fishing_vessel
3,3,warship
4,4,work_vessel


data.yaml: {'train': 'images/train', 'val': 'images/test', 'test': 'images/test', 'nc': 5, 'names': {0: 'small_boat', 1: 'cargo_ship', 2: 'fishing_vessel', 3: 'warship', 4: 'work_vessel'}}


**Interpretación:** la salida anterior identifica la raíz elegida y las versiones reales.
`val: images/test` describe el kit; no crea una validación independiente.
**Implicación:** conservar este manifiesto permite distinguir cambios de entorno de cambios en los datos [E, §4.3; L, cap. 5].


## 2. Inventario y calidad de imágenes y etiquetas

**Preguntas:** ¿está completa la entrega? ¿las etiquetas son legibles, válidas y emparejables?
**Método:** recorrer todos los archivos sin muestreo, verificar la estructura con Pillow y
decodificar los píxeles a resolución nativa. Registrar ancho, alto, canales, formato y modo original.
Validar cada línea YOLO: exactamente cinco números finitos, clase entera 0–4,
coordenadas normalizadas, tamaño positivo y bordes dentro de `[0,1] ± TOL`.
Una repetición exacta se conserva, se marca y se separa de los conteos exploratorios únicos.
No se arreglan etiquetas ni se ocultan filas inválidas: quedan en el registro de incidencias.

`vacía_candidata_negativa` significa ausencia de objetos anotados, **no** confirmación visual
de una negativa legítima. `ausente`, `ilegible`, `solo_inválidas`, `ambigua` y
`sin_etiquetas_por_diseño` son estados diferentes. Una anotación sin imagen es huérfana.
Los IDs repetidos en una partición se consideran ambiguos y no se usan para frecuencias.
Las etiquetas del oculto, si aparecieran, se registran como inesperadas sin leer su contenido.

Fundamento: inspección y calidad [L, caps. 2 y 5]; preparar los datos antes de entrenar [E, §4.3].


In [ ]:
def parse_yolo(text, tol=TOL):
    """Retorna filas válidas (incluidas repetidas), incidencias y número de líneas no vacías."""
    rows, issues, seen, n_lines = [], [], {}, 0
    for line_no, line in enumerate(text.splitlines(), 1):
        tokens = line.split()
        if not tokens:
            continue
        n_lines += 1
        problem = None
        values = None
        if len(tokens) != 5:
            problem = 'cantidad_valores'
        else:
            try:
                values = np.asarray([float(t) for t in tokens], dtype=float)
            except ValueError:
                problem = 'no_numérico'
        if values is not None:
            if not np.isfinite(values).all():
                problem = 'no_finito'
            else:
                c, xc, yc, w, h = values
                if c != int(c) or not 0 <= c < 5:
                    problem = 'clase_inválida'
                elif w <= 0 or h <= 0:
                    problem = 'tamaño_no_positivo'
                elif (np.array([xc, yc, w, h]) < -tol).any() or (np.array([xc, yc, w, h]) > 1 + tol).any():
                    problem = 'coordenadas_fuera_rango'
                elif min(xc - w/2, yc - h/2) < -tol or max(xc + w/2, yc + h/2) > 1 + tol:
                    problem = 'bordes_fuera_imagen'
        if problem:
            issues.append(dict(line=line_no, issue=problem, detail=line))
            continue
        key = tuple(values)
        duplicate = key in seen
        if duplicate:
            issues.append(dict(line=line_no, issue='anotación_duplicada', detail=f'Igual a línea {seen[key]}'))
        seen.setdefault(key, line_no)
        rows.append(dict(line=line_no, class_id=int(values[0]), xc=values[1], yc=values[2],
                         wn=values[3], hn=values[4], duplicate=duplicate))
    return rows, issues, n_lines

def listed_files(folder, extensions):
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in extensions)

def infer_source_group(stem):
    # Solo convención del nombre; la escena no puede confirmarse sin metadatos externos.
    prefix = stem.split('__', 1)[0].lower()
    source = prefix if prefix in {'fair1m', 'shiprsimagenet', 'vhrships', 'xview'} else 'desconocida'
    group = re.sub(r'__t\d+_\d+$', '', stem) if source != 'desconocida' else None
    return source, group

image_rows, label_rows, box_rows, issue_rows = [], [], [], []
started = time.perf_counter()
for split in SPLITS:
    image_dir, label_dir = DATA_ROOT / 'images' / split, DATA_ROOT / 'labels' / split
    if not image_dir.is_dir():
        issue_rows.append(dict(split=split, path=f'images/{split}', line=None,
                               issue='carpeta_imágenes_ausente', detail='Partición incompleta'))
    if split != 'oculto' and not label_dir.is_dir():
        issue_rows.append(dict(split=split, path=f'labels/{split}', line=None,
                               issue='carpeta_etiquetas_ausente', detail='Partición incompleta'))
    image_paths = listed_files(image_dir, IMAGE_EXT)
    label_paths = listed_files(label_dir, {'.txt'})
    images_by_id, labels_by_id = defaultdict(list), defaultdict(list)
    for path in image_paths:
        images_by_id[path.stem].append(path)
    for path in label_paths:
        labels_by_id[path.stem].append(path)
    for kind, index in [('imagen', images_by_id), ('etiqueta', labels_by_id)]:
        for image_id, paths in index.items():
            if len(paths) > 1:
                issue_rows.append(dict(split=split, path=image_id, line=None,
                    issue=f'id_{kind}_ambiguo', detail='; '.join(p.relative_to(DATA_ROOT).as_posix() for p in paths)))
    label_meta = {}
    for path in label_paths:
        rel = path.relative_to(DATA_ROOT).as_posix()
        matches = images_by_id.get(path.stem, [])
        meta = dict(split=split, image_id=path.stem, path=rel, sha256=None,
                    n_lines=None, n_valid=None, n_unique=None, n_invalid=None, n_duplicates=None)
        if split == 'oculto':
            meta['status'] = 'inesperada_oculto_no_leída'
            issue_rows.append(dict(split=split, path=rel, line=None, issue=meta['status'], detail='No se usa'))
        else:
            try:
                meta['sha256'] = sha256_file(path)
                valid, errors, n_lines = parse_yolo(path.read_text(encoding='utf-8-sig'))
                meta.update(n_lines=n_lines, n_valid=len(valid), n_unique=sum(not r['duplicate'] for r in valid),
                            n_invalid=n_lines-len(valid), n_duplicates=sum(r['duplicate'] for r in valid))
                meta['status'] = ('vacía_candidata_negativa' if n_lines == 0 else
                                  'solo_inválidas' if not valid else
                                  'con_errores' if n_lines > len(valid) else 'anotada')
                for row in valid:
                    box_rows.append(dict(split=split, image_id=path.stem, label_path=rel,
                        paired=len(matches) == 1 and len(labels_by_id[path.stem]) == 1, **row))
                for row in errors:
                    issue_rows.append(dict(split=split, path=rel, **row))
            except (OSError, UnicodeError) as error:
                meta['status'] = 'ilegible'
                issue_rows.append(dict(split=split, path=rel, line=None, issue='etiqueta_ilegible', detail=str(error)))
            if not matches:
                issue_rows.append(dict(split=split, path=rel, line=None, issue='etiqueta_huérfana', detail='Sin imagen'))
        label_rows.append(meta)
        if len(labels_by_id[path.stem]) == 1:
            label_meta[path.stem] = meta
    for i, path in enumerate(image_paths, 1):
        rel = path.relative_to(DATA_ROOT).as_posix()
        source, group = infer_source_group(path.stem)
        meta = dict(split=split, image_id=path.stem, path=rel, bytes=path.stat().st_size,
                    width=None, height=None, channels=None, mode=None, format=None,
                    readable=False, sha256=None, pixel_sha256=None, source=source, group_candidate=group)
        try:
            if HASH_IMAGES:
                meta['sha256'] = sha256_file(path)
            with Image.open(path) as im:
                meta.update(width=im.width, height=im.height, channels=len(im.getbands()),
                            mode=im.mode, format=im.format)
                im.verify()
            with Image.open(path) as im:
                im.load()  # Fuerza la decodificación completa: cabecera válida no basta.
                if HASH_IMAGES:
                    pixel_hash = hashlib.sha256(f'{im.mode}|{im.size}|'.encode())
                    pixel_hash.update(im.tobytes())
                    meta['pixel_sha256'] = pixel_hash.hexdigest()
            meta['readable'] = True
            if meta['mode'] != 'RGB' or meta['channels'] != 3:
                issue_rows.append(dict(split=split, path=rel, line=None, issue='modo_no_RGB', detail=str(meta['mode'])))
        except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as error:
            issue_rows.append(dict(split=split, path=rel, line=None, issue='imagen_ilegible', detail=str(error)))
        annotation = label_meta.get(path.stem)
        if split == 'oculto':
            meta['label_status'] = 'sin_etiquetas_por_diseño'
        elif len(images_by_id[path.stem]) != 1 or len(labels_by_id.get(path.stem, [])) > 1:
            meta['label_status'] = 'ambigua'
        elif annotation is None:
            meta['label_status'] = 'ausente'
            issue_rows.append(dict(split=split, path=rel, line=None, issue='etiqueta_ausente', detail='No asumir negativa'))
        else:
            meta['label_status'] = annotation['status']
        meta['n_objects_unique'] = (annotation['n_unique'] if annotation and split != 'oculto'
                                    and meta['label_status'] != 'ambigua' else None)
        image_rows.append(meta)
        if i % 1000 == 0:
            print(f'{split}: {i}/{len(image_paths)} imágenes verificadas')

images = pd.DataFrame(image_rows, columns=[
    'split','image_id','path','bytes','width','height','channels','mode','format','readable',
    'sha256','pixel_sha256','source','group_candidate','label_status','n_objects_unique'])
labels = pd.DataFrame(label_rows, columns=[
    'split','image_id','path','sha256','n_lines','n_valid','n_unique','n_invalid','n_duplicates','status'])
boxes_raw = pd.DataFrame(box_rows, columns=[
    'split','image_id','label_path','paired','line','class_id','xc','yc','wn','hn','duplicate'])
issues = pd.DataFrame(issue_rows, columns=['split','path','line','issue','detail'])
if images.empty:
    raise RuntimeError('No hay imágenes reconocidas en la raíz seleccionada.')
if images.duplicated(['split', 'image_id']).any():
    safe_images = images.loc[~images.duplicated(['split', 'image_id'], keep=False)]
else:
    safe_images = images
valid_boxes = boxes_raw.loc[boxes_raw['paired'].eq(True) & boxes_raw['duplicate'].eq(False)].copy()
valid_boxes = valid_boxes.merge(safe_images[['split','image_id','readable','width','height','source']],
                                on=['split','image_id'], how='left', validate='many_to_one')
boxes = valid_boxes.loc[valid_boxes['readable'].eq(True)].copy()
boxes['class_name'] = boxes['class_id'].map(dict(enumerate(CLASS_NAMES)))
for column in ['width', 'height']:
    images[column] = pd.to_numeric(images[column])
images['aspect_ratio'] = images['width'] / images['height']
images['image_area_px'] = images['width'] * images['height']
safe_images = images.loc[~images.duplicated(['split', 'image_id'], keep=False)].copy()

summary_rows = []
for split in SPLITS:
    im, lab = images.loc[images.split.eq(split)], labels.loc[labels.split.eq(split)]
    annotated = split != 'oculto'
    summary_rows.append(dict(
        split=split, images=len(im), labels=len(lab), corrupt=int((~im.readable).sum()),
        annotation_lines=lab.n_lines.sum(min_count=1) if annotated else np.nan,
        valid_lines=lab.n_valid.sum(min_count=1) if annotated else np.nan,
        unique_valid_labels=lab.n_unique.sum(min_count=1) if annotated else np.nan,
        analysis_objects=int(boxes.split.eq(split).sum()) if annotated else np.nan,
        missing_labels=int(im.label_status.eq('ausente').sum()) if annotated else np.nan,
        empty_labels=int(im.label_status.eq('vacía_candidata_negativa').sum()) if annotated else np.nan,
        only_invalid=int(im.label_status.eq('solo_inválidas').sum()) if annotated else np.nan,
        images_without_valid_objects=int(im.n_objects_unique.eq(0).sum()) if annotated else np.nan,
        orphan_labels=int(((issues.split == split) & (issues.issue == 'etiqueta_huérfana')).sum()),
        expected_images=EXPECTED[split][0], expected_objects=EXPECTED[split][1]))
summary = pd.DataFrame(summary_rows)
summary['delta_images'] = summary['images'] - summary.expected_images
summary['delta_lines'] = summary.annotation_lines - summary.expected_objects
display(summary)
display(images.groupby(['split','format','mode','channels'], dropna=False).size().rename('imágenes').to_frame())
display(pd.crosstab(images.split, images.label_status))
display(issues.groupby(['split','issue']).size().rename('incidencias').to_frame() if not issues.empty else
        pd.DataFrame({'estado': ['Sin inconsistencias detectadas por estas reglas']}))
display(issues.head(30))
print(f'Auditoría completa en {time.perf_counter()-started:.1f} s. Registro íntegro: issues.csv')


In [ ]:
inventory_findings = []
for row in summary.itertuples():
    inventory_findings.append(
        f'- **{row.split}:** {row.images:,} imágenes ({row.delta_images:+,} frente a referencia), '
        f'{row.labels:,} etiquetas y {row.corrupt:,} imágenes ilegibles.'
        + (' Objetos reales desconocidos: partición sin etiquetas.' if row.split == 'oculto' else
           f' {row.annotation_lines:g} líneas anotadas; {row.analysis_objects:g} cajas únicas, emparejadas y legibles para análisis.'))
display(Markdown('### Interpretación calculada del inventario\n' + '\n'.join(inventory_findings) +
    f'\n\nSe registraron **{len(issues):,} incidencias**; no equivalen necesariamente a archivos distintos. '
    'Las diferencias con las cifras oficiales requieren revisión; no justifican eliminar datos. '
    'Una etiqueta vacía necesita revisión humana antes de declararse negativa legítima. '
    '**Implicación:** resolver y documentar inconsistencias antes de fijar el protocolo de entrenamiento.'))


## 3. Comprobación visual de imágenes y cajas

**Pregunta:** ¿las cajas normalizadas se proyectan correctamente sobre imágenes nativas?
**Método:** muestras deterministas de `train`: una imagen por clase, escena densa,
imagen con caja de menor área relativa y candidata negativa si existe.
Pillow lee las imágenes originales; OpenCV comprueba, en un ejemplo, la concordancia BGR→RGB.
Las cajas se dibujan con Matplotlib en coordenadas originales, sin guardar imágenes transformadas.
Los casos atípicos no son una muestra representativa de toda la población.
Referencia: [L], cap. 2. La inspección de cajas es una adaptación para SpaceLAB.


In [ ]:
analysis_images = safe_images.loc[safe_images.split.isin(EDA_SPLITS) & safe_images.readable].copy()
analysis_boxes = boxes.loc[boxes.split.isin(EDA_SPLITS)].copy()
train_images = analysis_images.loc[analysis_images.split.eq('train')]
train_boxes = analysis_boxes.loc[analysis_boxes.split.eq('train')]
palette = dict(enumerate(sns.color_palette('tab10', 5)))
selected = []
for class_id in range(5):
    candidates = train_boxes.loc[train_boxes.class_id.eq(class_id), 'image_id'].drop_duplicates().sort_values()
    if len(candidates):
        image_id = candidates.sample(1, random_state=SEED + class_id).iloc[0]
        selected.append((image_id, CLASS_NAMES[class_id]))
if len(train_boxes):
    selected.append((train_boxes.groupby('image_id').size().idxmax(), 'Mayor densidad'))
    relative_area = train_boxes.wn * train_boxes.hn
    selected.append((train_boxes.loc[relative_area.idxmin(), 'image_id'], 'Menor caja relativa'))
negative = train_images.loc[train_images.label_status.eq('vacía_candidata_negativa')]
if len(negative):
    selected.append((negative.sort_values('image_id').iloc[0].image_id, 'Vacía: negativa sin confirmar'))
if selected:
    fig, axes = plt.subplots((len(selected) + 2)//3, 3, figsize=(16, 5*((len(selected)+2)//3)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (image_id, reason) in zip(axes.flat, selected):
        row = train_images.loc[train_images.image_id.eq(image_id)].iloc[0]
        with Image.open(DATA_ROOT / row.path) as im:
            ax.imshow(im)
        group = train_boxes.loc[train_boxes.image_id.eq(image_id)]
        for box in group.itertuples():
            ax.add_patch(Rectangle(((box.xc-box.wn/2)*row.width, (box.yc-box.hn/2)*row.height),
                box.wn*row.width, box.hn*row.height, fill=False, color=palette[box.class_id], linewidth=1))
        ax.set_title(f'{reason} | {len(group)} cajas\n{image_id}', fontsize=9)
    save_figure('03_inspeccion_visual')
    example = train_images.loc[train_images.image_id.eq(selected[0][0])].iloc[0]
    # imdecode admite rutas con caracteres Unicode en Windows.
    bgr = cv2.imdecode(np.fromfile(DATA_ROOT / example.path, dtype=np.uint8), cv2.IMREAD_COLOR)
    with Image.open(DATA_ROOT / example.path) as im:
        rgb = np.asarray(im.convert('RGB'))
    if bgr is not None and bgr.shape == rgb.shape:
        error = np.abs(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(float) - rgb.astype(float))
        print('Pillow vs OpenCV en un ejemplo: diferencia absoluta media =', error.mean())
    else:
        print('OpenCV no pudo decodificar el ejemplo con la misma forma; revisar manualmente.')
else:
    print('No hay ejemplos válidos en train para inspección visual.')


**Interpretación y límites:** comprobar visualmente ajuste de cajas, colores y legibilidad.
El notebook no declara que todas las etiquetas sean correctas a partir de una muestra.
**Implicación:** los errores sistemáticos observados deben registrarse antes de proponer limpieza;
la automatización no confirma etiquetas ausentes ni negativas legítimas.


## 4. Distribución de clases y coocurrencia

**Preguntas:** ¿qué clases tienen menor cobertura? ¿aparecen juntas y en cuántas imágenes?
**Método:** contar cajas válidas, únicas, emparejadas y con imagen legible. Frecuencias separadas
por partición seleccionada; no agregar `test` al entrenamiento. La distribución por imagen incluye
ceros **solo** en imágenes con etiqueta completamente válida o vacía, excluyendo ausentes,
ambiguas e inválidas. Los positivos válidos de etiquetas con errores sí contribuyen al conteo
de objetos, pero no al denominador de presencia por imagen.
La coocurrencia usa presencia binaria: la diagonal cuenta imágenes con esa clase.
El cociente máxima/mínima se declara infinito si alguna clase tiene cero instancias.

**Desbalance no es sinónimo de sesgo:** el primero es una diferencia de frecuencias;
sesgo de selección o medición exige evidencia sobre muestreo, población y anotación [L, cap. 5].
Las cinco clases pesan igual en las métricas macro del reto [E, §4.1; documento oficial].


In [ ]:
class_tables, per_image_tables, cooccurrences = [], [], {}
class_findings = []
for split in EDA_SPLITS:
    subset = analysis_boxes.loc[analysis_boxes.split.eq(split)]
    counts = subset.groupby('class_id').size().reindex(range(5), fill_value=0)
    eligible = analysis_images.loc[analysis_images.split.eq(split) &
        analysis_images.label_status.isin(['anotada', 'vacía_candidata_negativa'])]
    per_image = pd.crosstab(subset.image_id, subset.class_id).reindex(
        index=eligible.image_id, columns=range(5), fill_value=0).astype(int)
    presence = (per_image > 0).astype(int)
    cooc = presence.T @ presence
    cooc.index = cooc.columns = CLASS_NAMES
    cooccurrences[split] = cooc
    total = int(counts.sum())
    table = pd.DataFrame({'split': split, 'class_id': range(5), 'class_name': CLASS_NAMES,
        'objects': counts.to_numpy(), 'percent_objects': 100*counts.to_numpy()/total if total else np.nan,
        'images_with_class': presence.sum().to_numpy(), 'eligible_images': len(eligible)})
    table['percent_images'] = 100*table.images_with_class/len(eligible) if len(eligible) else np.nan
    class_tables.append(table)
    per_image.columns = CLASS_NAMES
    display(table)
    display(per_image.describe(percentiles=[.5, .9, .95, .99]).T)
    per_image_tables.append(per_image.reset_index().assign(split=split))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(CLASS_NAMES, table.objects, color=list(palette.values()))
    axes[0].set(ylabel='Objetos válidos únicos', title=f'{split}: frecuencia absoluta')
    axes[1].bar(CLASS_NAMES, table.percent_objects, color=list(palette.values()))
    axes[1].set(ylabel='% de objetos', title=f'{split}: participación')
    for ax in axes:
        ax.tick_params(axis='x', rotation=30)
    save_figure(f'04_clases_{split}')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if len(per_image):
        axes[0].boxplot([per_image[c].values for c in CLASS_NAMES], showfliers=True)
        axes[0].set_xticks(range(1, 6), CLASS_NAMES)
        axes[0].set_yscale('symlog', linthresh=1)
        axes[0].tick_params(axis='x', rotation=30)
        axes[0].set(title=f'{split}: instancias por imagen, incluidos ceros', ylabel='Instancias (escala symlog)')
        sns.heatmap(cooc, annot=True, fmt='d', cmap='Blues', ax=axes[1])
        axes[1].set_title(f'{split}: coocurrencia por imagen')
    else:
        axes[0].text(.1, .5, 'Sin imágenes con etiquetas completas')
    save_figure(f'04_presencia_{split}')
    ratio = float(counts.max()/counts.min()) if counts.min() else float('inf')
    work = table.iloc[4]
    class_findings.append(f'- **{split}:** razón máxima/mínima = **{ratio:.2f}**; '
        f'`work_vessel` aporta **{int(work.objects):,} objetos ({work.percent_objects:.2f}%)** y aparece '
        f'en **{int(work.images_with_class):,}/{len(eligible):,}** imágenes con etiqueta completa. '
        f'Clase(s) menos frecuente(s): {", ".join(CLASS_NAMES[c] for c in counts[counts.eq(counts.min())].index)}.')
class_frequency = pd.concat(class_tables, ignore_index=True)
class_per_image = pd.concat(per_image_tables, ignore_index=True)
display(Markdown('### Interpretación calculada\n' + '\n'.join(class_findings) +
    '\n\n**Implicación:** comparar cobertura de la clase minoritaria y coocurrencias antes de '
    'considerar balanceo. No se aplica sobremuestreo ni ponderación; los conteos no prueban '
    'sesgo ni permiten predecir qué clase tendrá peor rendimiento.'))


## 5. Dimensiones y resolución originales

**Preguntas:** ¿qué geometrías predominan? ¿qué pérdida de escala podría causar una entrada fija?
**Método:** estadísticas por partición de ancho, alto, razón `ancho/alto` y área.
Cuadrada significa `ancho == alto`, horizontal `ancho > alto` y vertical `ancho < alto`.
La tabla de resoluciones conserva todas las combinaciones; la gráfica muestra las 15 más comunes.
La desviación estándar es muestral (`ddof=1`); es indefinida si solo hay una imagen.
Se excluyen imágenes ilegibles y se indica el denominador. No se redimensiona ninguna imagen.
Referencia: [L], cap. 6, tamaño y resolución; [E], §4.3.2, preparación.


In [ ]:
dimension_columns = ['width', 'height', 'aspect_ratio', 'image_area_px']
dimension_stats = analysis_images.groupby('split')[dimension_columns].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99])
resolutions = analysis_images.groupby(['split','width','height']).size().rename('images').reset_index()
resolutions = resolutions.sort_values(['split','images'], ascending=[True,False])
analysis_images['geometry'] = np.select(
    [analysis_images.width.eq(analysis_images.height), analysis_images.width.gt(analysis_images.height)],
    ['cuadrada', 'horizontal'], default='vertical')
geometry = pd.crosstab(analysis_images.split, analysis_images.geometry).reindex(
    columns=['cuadrada','horizontal','vertical'], fill_value=0)
display(dimension_stats)
display(resolutions.head(20))
display(geometry)
for split in EDA_SPLITS:
    subset = analysis_images.loc[analysis_images.split.eq(split)]
    if subset.empty:
        continue
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for column, ax in zip(['width','height','image_area_px','aspect_ratio'], axes.flat):
        ax.hist(subset[column], bins=30, color='steelblue', edgecolor='white')
        ax.set(title=column, ylabel='Imágenes', xlabel='píxeles' if column in ('width','height') else
               'píxeles²' if column == 'image_area_px' else 'ancho / alto')
    axes[1,1].scatter(subset.width, subset.height, s=10, alpha=.25)
    axes[1,1].set(xlabel='Ancho (px)', ylabel='Alto (px)', title=f'{split}: dimensiones nativas')
    top = resolutions.loc[resolutions.split.eq(split)].head(15)
    axes[1,2].barh([f'{int(w)}×{int(h)}' for w,h in zip(top.width,top.height)], top.images)
    axes[1,2].set(xlabel='Imágenes', title='Resoluciones más frecuentes')
    axes[1,2].invert_yaxis()
    save_figure(f'05_dimensiones_{split}')
    counts = geometry.loc[split]
    dominant = ', '.join(counts[counts.eq(counts.max())].index)
    display(Markdown(f'**Interpretación calculada — {split}:** {len(subset):,} imágenes legibles; '
        f'ancho {subset.width.min():g}–{subset.width.max():g} px, alto '
        f'{subset.height.min():g}–{subset.height.max():g} px. Geometría predominante: **{dominant}** '
        f'({counts.max():,} imágenes). Mediana de aspecto: {subset.aspect_ratio.median():.3f}. '
        '**Implicación:** una entrada fija puede reducir detalle; estirar a un cuadrado altera '
        'proporciones cuando el aspecto difiere de 1. Letterbox preservaría aspecto pero usa padding. '
        'La conveniencia de cada alternativa requiere experimentos posteriores.'))


## 6. Escala de cajas, densidad y distribución espacial

**Preguntas:** ¿qué tamaño tienen los objetos y cuántos comparten una imagen?
**Método:** ancho/alto en píxeles nativos, área relativa y área de caja. Umbrales descriptivos
de área `<32²`, `32²–96²`, `≥96²` en píxeles nativos, sin presentarlos como una evaluación COCO.
Calcular aritméticamente la escala que produciría ajustar el lado largo a 640/1024:
`factor = lado_objetivo / max(ancho, alto)`. No se generan imágenes nuevas, tiles ni augmentations.
Estos cálculos no simulan padding, interpolación ni rendimiento del detector.
El histograma de densidad incluye ceros confirmados por archivos vacíos, pero no etiquetas ausentes.
La matriz espacial usa centros normalizados. Las cajas horizontales no permiten inferir el rumbo
de un barco: su razón de aspecto no es una orientación real.
Análisis adaptado a detección; fundamento general [L, caps. 5–6].


In [ ]:
analysis_boxes['box_width_px'] = analysis_boxes.wn * analysis_boxes.width
analysis_boxes['box_height_px'] = analysis_boxes.hn * analysis_boxes.height
analysis_boxes['box_area_px'] = analysis_boxes.box_width_px * analysis_boxes.box_height_px
analysis_boxes['relative_area'] = analysis_boxes.wn * analysis_boxes.hn
analysis_boxes['native_area_bin'] = pd.cut(analysis_boxes.box_area_px,
    bins=[0,32**2,96**2,np.inf], right=False, labels=['<32²','32²–96²','≥96²'])
scale_columns = ['box_width_px','box_height_px','box_area_px','relative_area']
box_stats = analysis_boxes.groupby(['split','class_name'])[scale_columns].describe(percentiles=[.05,.5,.95])
display(box_stats)
display(pd.crosstab([analysis_boxes.split, analysis_boxes.class_name], analysis_boxes.native_area_bin))
scale_rows = []
for split in EDA_SPLITS:
    subset = analysis_boxes.loc[analysis_boxes.split.eq(split)]
    if subset.empty:
        continue
    for target in (640,1024):
        factor = target / subset[['width','height']].max(axis=1)
        short_side = np.minimum(subset.box_width_px, subset.box_height_px) * factor
        scale_rows.append(dict(split=split, target_long_side=target, n_objects=len(subset),
            pct_images_downscaled=100*(analysis_images.loc[analysis_images.split.eq(split), ['width','height']].max(axis=1)>target).mean(),
            median_short_side_px=short_side.median(), pct_short_side_under_4px=100*short_side.lt(4).mean(),
            pct_short_side_under_8px=100*short_side.lt(8).mean()))
    density = analysis_images.loc[analysis_images.split.eq(split) & analysis_images.label_status.isin(
        ['anotada','vacía_candidata_negativa']), ['image_id','n_objects_unique']].copy()
    display(density.n_objects_unique.describe(percentiles=[.5,.9,.95,.99]).to_frame('objetos/imagen'))
    fig, axes = plt.subplots(1, 3, figsize=(16,4))
    sns.histplot(data=subset, x='relative_area', hue='class_name', bins=40, log_scale=True,
                 element='step', ax=axes[0])
    axes[0].set(xlabel='Área caja / área imagen (log)', title='Escala relativa')
    axes[1].hist(density.n_objects_unique, bins=40)
    axes[1].set(xlabel='Objetos válidos únicos', ylabel='Imágenes', title='Densidad por imagen')
    histogram = axes[2].hist2d(subset.xc, subset.yc, bins=20, range=[[0,1],[0,1]], cmap='magma')
    axes[2].invert_yaxis()
    axes[2].set(xlabel='Centro x normalizado', ylabel='Centro y normalizado', title='Centros de cajas')
    fig.colorbar(histogram[3], ax=axes[2], label='Objetos')
    save_figure(f'06_escala_densidad_{split}')
    display(Markdown(f'**Interpretación calculada — {split}:** mediana del área relativa '
        f'**{100*subset.relative_area.median():.4f}%**; '
        f'**{100*subset.relative_area.lt(.0005).mean():.2f}%** de cajas ocupan menos del 0.05% '
        f'de la imagen. Máxima densidad con etiqueta completa: '
        f'**{density.n_objects_unique.max():g} objetos/imagen**. '
        '**Implicación:** contrastar resolución y posibles recortes en una etapa posterior; '
        'la evidencia geométrica no demuestra por sí sola una mejora de mAP.'))
scale_projection = pd.DataFrame(scale_rows)
display(scale_projection)


## 7. Procedencia, duplicados y posibles fugas

**Preguntas:** ¿se repiten imágenes o posibles escenas entre particiones? ¿la cobertura varía por fuente?
**Método:** SHA-256 del archivo y de píxeles decodificados nativos (incluyendo modo y dimensiones).
Esto detecta igualdad exacta; no detecta todas las imágenes similares, reescaladas, rotadas o recortes
solapados. Un hash distinto no prueba independencia. No se hace búsqueda de etiquetas externas.
El prefijo del nombre sugiere fuente; retirar únicamente `__tX_Y` final genera un **candidato**
de escena, nunca una escena confirmada. Se comparan train/test y se mantiene como desconocido
el origen de los nombres anonimizados del oculto. No se emplean hashes para inferir sus clases.

Los grupos inferidos y diferencias por fuente aportan evidencia para diseñar después la validación
[E, §4.3.1; L, cap. 5]. La propuesta de análisis por segmentos se relaciona con [L, cap. 8],
pero aquí no hay métricas de rendimiento ni conclusiones sobre equidad poblacional.


In [ ]:
duplicate_tables = []
for column in ['sha256','pixel_sha256']:
    hashed = images.dropna(subset=[column])
    duplicate = hashed.loc[hashed.duplicated(column, keep=False), ['split','image_id','path',column]].copy()
    duplicate['hash_type'] = column
    duplicate.rename(columns={column:'hash'}, inplace=True)
    if len(duplicate):
        duplicate['cross_split'] = duplicate.groupby('hash')['split'].transform('nunique').gt(1)
    else:
        duplicate['cross_split'] = pd.Series(dtype=bool)
    duplicate_tables.append(duplicate)
duplicates = pd.concat(duplicate_tables, ignore_index=True)
display(duplicates.head(30))
print('Hashes:', 'completos (excepto errores registrados)' if HASH_IMAGES else 'DESACTIVADOS: análisis de duplicados incompleto')
source_coverage = pd.crosstab(images.split, images.source)
display(source_coverage)
source_class = pd.crosstab(analysis_boxes.source, analysis_boxes.class_name).reindex(columns=CLASS_NAMES, fill_value=0)
source_class_pct = source_class.div(source_class.sum(axis=1).replace(0,np.nan), axis=0)*100
display(source_class)
if len(source_class):
    fig, axes = plt.subplots(1,2,figsize=(15,4))
    sns.heatmap(source_class, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_title('Objetos por fuente inferida y clase (train por defecto)')
    sns.heatmap(source_class_pct, annot=True, fmt='.1f', cmap='YlGnBu', ax=axes[1])
    axes[1].set_title('% dentro de cada fuente inferida')
    save_figure('07_fuente_clase')
known_groups = images.dropna(subset=['group_candidate']).copy()
group_splits = known_groups.groupby('group_candidate')['split'].agg(lambda s: '|'.join(sorted(set(s))))
shared = group_splits[group_splits.str.contains('train') & group_splits.str.contains('test')]
shared_groups = known_groups.loc[known_groups.group_candidate.isin(shared.index),
    ['split','image_id','path','source','group_candidate']].sort_values(['group_candidate','split'])
display(shared_groups.head(30))
cross_hash_groups = duplicates.loc[duplicates.cross_split].groupby('hash_type')['hash'].nunique().to_dict()
display(Markdown('### Interpretación calculada\n'
    f'- Grupos de hashes con miembros en varias particiones: **{cross_hash_groups}**.\n'
    f'- Candidatos de escena compartidos entre train/test: **{len(shared):,}**.\n'
    f'- Imágenes de fuente desconocida: **{int(images.source.eq("desconocida").sum()):,}**.\n\n'
    '**Implicación:** revisar los registros antes de fijar grupos de validación. '
    'Compartir un candidato de nombre requiere verificación y compartir fuente no significa '
    'compartir escena. Si no hay coincidencias exactas, aún pueden existir escenas relacionadas. '
    'Las diferencias de cobertura son descriptivas; sin metadatos de adquisición no se demuestra '
    'representatividad geográfica, temporal o de sensores. No se crean particiones.'))


## 8. Exportación, evidencia y decisiones pendientes

**Pregunta:** ¿puede otro compañero reconstruir los resultados y distinguir datos de hipótesis?
**Método:** guardar tablas completas, figuras, resumen y manifiesto fuera de los originales.
Las huellas combinan rutas relativas y hashes; no dependen de la letra de unidad o ubicación de Drive.
Los inventarios incluyen cada archivo auditado; el manifiesto registra la huella de su contenido.
Se comprueba que los originales mantengan tamaño/fecha de modificación desde el inicio de la
fase de exportación; las lecturas anteriores nunca abren archivos para escritura.
Las tablas pueden contener rutas y nombres del dataset; revisar antes de compartir informes.


In [ ]:
def content_fingerprint(frame, columns):
    digest = hashlib.sha256()
    for row in frame.sort_values('path')[columns].itertuples(index=False, name=None):
        digest.update((json.dumps(row, ensure_ascii=False, default=str) + '\n').encode('utf-8'))
    return digest.hexdigest()

tables = {
    'inventory_images': images, 'inventory_labels': labels, 'boxes_valid_including_duplicates': boxes_raw,
    'boxes_analysis': analysis_boxes, 'issues': issues, 'summary': summary,
    'class_frequency': class_frequency, 'class_per_image': class_per_image,
    'dimension_stats': dimension_stats, 'resolutions': resolutions, 'geometry': geometry,
    'box_stats': box_stats, 'scale_projection': scale_projection, 'duplicates': duplicates,
    'source_coverage': source_coverage, 'source_class': source_class, 'shared_group_candidates': shared_groups,
}
tables.update({f'cooccurrence_{split}': table for split,table in cooccurrences.items()})
try:
    revision = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],
        capture_output=True, text=True, check=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    revision = None
manifest = dict(
    created_utc=datetime.now(timezone.utc).isoformat(), seed=SEED, tolerance=TOL,
    python=platform.python_version(), versions=VERSIONS, git_revision=revision,
    dataset_root=str(DATA_ROOT), examined_paths=[str(p) for p in EXAMINED],
    analysis_splits=list(EDA_SPLITS), structural_audit_splits=list(SPLITS), hashes_enabled=HASH_IMAGES,
    yaml_sha256=sha256_file(DATA_ROOT/'data.yaml'),
    image_content_fingerprint=content_fingerprint(images, ['path','bytes','sha256','pixel_sha256']),
    label_content_fingerprint=content_fingerprint(labels, ['path','sha256']),
    fingerprint_note='Si HASH_IMAGES=False, la huella de imágenes no valida su contenido.',
    notebook_sha256=sha256_file(REPO_ROOT/'notebooks/01_EDA_SpaceLAB.ipynb')
        if (REPO_ROOT/'notebooks/01_EDA_SpaceLAB.ipynb').is_file() else None,
)
findings = '\n'.join(inventory_findings + class_findings)
pending = [
    ('Validación', f'{len(shared)} candidatos de escena train/test; revisar shared_group_candidates.csv',
     'Confirmar grupos/metadatos y acordar validación dentro de train; conservar test sin tuning.'),
    ('Arquitecturas', 'Escala, densidad y clases medidas; sin entrenamientos ni predicciones',
     'Comparar baselines con protocolo y presupuesto comunes; aún no elegir ganador.'),
    ('Resolución / recortes', 'Ver box_stats.csv, scale_projection.csv y figuras de escala',
     'Diseñar comparación posterior; las proyecciones geométricas no son resultados de un detector.'),
    ('Preprocesamiento', f'{len(issues)} incidencias en issues.csv',
     'Revisar errores; no aplicar filtros ni correcciones generales por defecto.'),
    ('Balance / augmentation', 'Ver frecuencias, coocurrencia y cobertura por fuente',
     'Formular hipótesis controladas; no sobremuestrear ni generar imágenes en esta etapa.'),
    ('Fuga / sobreajuste', f'Coincidencias entre particiones: {cross_hash_groups}',
     'Independencia no garantizada por hashes distintos; medir sobreajuste cuando existan modelos.'),
]
decision_evidence = pd.DataFrame(pending, columns=['decisión_futura','evidencia','pendiente'])
display(decision_evidence)
if EXPORT:
    originals = [DATA_ROOT/p for p in images.path] + [DATA_ROOT/p for p in labels.path] + [DATA_ROOT/'data.yaml']
    before = {str(p): (p.stat().st_size,p.stat().st_mtime_ns) for p in originals}
    for name, table in tables.items():
        table.to_csv(OUT / f'{name}.csv', index=not isinstance(table.index, pd.RangeIndex), encoding='utf-8')
    decision_evidence.to_csv(OUT/'decision_evidence.csv', index=False, encoding='utf-8')
    (OUT/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    (OUT/'findings.md').write_text('# Hallazgos calculados\n\n' + findings +
        '\n\nSin entrenamiento, limpieza automática, generación de particiones ni ajuste de umbrales.\n', encoding='utf-8')
    assert all(before[str(p)] == (p.stat().st_size,p.stat().st_mtime_ns) for p in originals), 'Un original cambió durante exportación.'
    print('Informes locales:', OUT)
else:
    print('EXPORT=False: resultados únicamente en memoria y salidas del notebook.')
display(Markdown('**Etapa completada:** evidencia descriptiva y registros auditables. '
    'Las decisiones de validación, modelo y entrenamiento siguen pendientes de revisión del equipo.'))


## Bibliografía y alcance de las atribuciones

- **[E]** Mohamed Elgendy (2020). *Deep Learning for Vision Systems*. Manning Publications.
  Cap. 4, §§4.1–4.4: métricas, baseline, preparación, partición y evaluación.
  [Editorial](https://www.manning.com/books/deep-learning-for-vision-systems) ·
  [Capítulo 4](https://livebook.manning.com/book/deep-learning-for-vision-systems/chapter-4).
- **[L]** Valliappa Lakshmanan, Martin Görner y Ryan Gillard (2021).
  *Practical Machine Learning for Computer Vision: End-to-End Machine Learning for Images*.
  O'Reilly Media. Cap. 2 (lectura y visualización), cap. 5 (creación, sesgos y partición),
  cap. 6 (preprocesamiento y resolución), cap. 8 (calidad y evaluación por segmentos).
  [Índice editorial](https://www.oreilly.com/library/view/practical-machine-learning/9781098102357/) ·
  [Cap. 5](https://www.oreilly.com/library/view/practical-machine-learning/9781098102357/ch05.html) ·
  [Cap. 6](https://www.oreilly.com/library/view/practical-machine-learning/9781098102357/ch06.html).
- **[R]** Organización SpaceLAB, FAC–PUJ / GeoVision Colombia (2026).
  [Documento oficial del reto](../kit_participantes/reto_spacelab.pdf), pp. 3–10;
  [YOLO](../kit_participantes/baseline_colab.ipynb),
  [Faster R-CNN](../kit_participantes/baseline_frcnn_colab.ipynb) y
  [evaluador oficial](../kit_participantes/evaluar.py).

Los temas de los libros orientan el protocolo. Las reglas de clases, particiones y puntaje
proceden del reto; las estadísticas provienen únicamente de los archivos leídos.
No se atribuyen a los autores el uso específico de hashes, candidatos por nombre, validación YOLO,
umbrales descriptivos de tamaño ni las proyecciones geométricas implementadas aquí.

**Límites:** no se mide rendimiento, no se confirma manualmente cada etiqueta, los hashes exactos
no son una búsqueda exhaustiva de similitud y las fuentes por nombre no reemplazan metadatos.
Antes de publicar en Git, limpiar las salidas del notebook: pueden incrustar imágenes aunque
los JPG estén en `.gitignore`. Los CSV, figuras y notebook ejecutado se guardan en `reports/eda/`,
carpeta ignorada. Conservar el notebook fuente sin salidas en el repositorio.
